In [ ]:

from scipy.spatial import Delaunay

# Generate vectors form Origin-Destination/network adjacency matrix for defined spatial regions 
## e. g. Parliment wards, cities, manucipalities, micro-regions

In [2]:
def generate_vector_for_defined_regions(adj_matrix, gdf, coordinates, method):
    
    #generate number of points on the boundary
    points_list= find_n_points_on_boundry(gdf, n=100)
    
        
    vecs = create_vectors(adj_matrix, coordinates, method=method)


    # add points on the boundary
    coords, vecs= add_boundry_points_to_coords_vecs(coordinates, vecs, points_list)

    #Interpolate missing vectors (excluding points on the boundary)
    vecs = interpolate_zero_vectors(coords, vecs, points_list)

    #remove the boundry points

    coords, vecs = remove_added_boundary_points_from_coords_vecs(coords, vecs, points_list)
    
    return vecs , coords
       

def create_vectors(adjacency, coordinates, method='sum'):
    """
    Create vectors for each location using adjacency matrix.

    Parameters:
        adjacency (ndarray): Adjacency matrix (n x n).
        coordinates (ndarray): Coordinate points (n x 2).
        method (str): 'sum' (default) or 'mean' for vector calculation.

    Returns:
        ndarray: Computed vectors (n x 2).
    """
    n = len(adjacency)
    vecs = np.zeros((n, 2))
    weight = np.zeros(n)

    for i in range(n):
        for j in range(n):
            new_vec = coordinates[j] - coordinates[i]
            vecs[i] += adjacency[i, j] * new_vec
            weight[i] += adjacency[i, j]

    if method == 'mean':
        for i in range(n):
            if weight[i] != 0:
                vecs[i] = vecs[i] / weight[i]

    return vecs

    
def find_n_points_on_boundry(gdf, n=100):
    #Load the shapefile of the boundary

    #assuming there's only one geometry in the GeoDataFrame as it is boundary line
    geometry = gdf.geometry.iloc[0]

    boundary = geometry.boundary
    boundary_length = boundary.length

    distances = np.linspace(0, boundary_length, n)

    #generate n points along the boundary
    points = [boundary.interpolate(distance) for distance in distances]
    points_list = [(point.x, point.y) for point in points]
    
    return points_list
    
    
def add_boundry_points_to_coords_vecs(coords, vecs, points_list):
    
    #make an array from coords and vecs
    coords = np.array(coords)
    vecs = np.array(vecs)
    
    for p in points_list:
        #cell_x = min(math.floor((p[0] - min_map_x) / l) + 1, w - 2)
        #cell_y = min(math.floor((p[1] - min_map_y) / l) + 1, h - 2)
        coords = np.append(coords, [[p[0], p[1]]], axis=0)
        vecs = np.append(vecs, [[0, 0]], axis=0)
        
        
    return coords, vecs





def remove_added_boundary_points_from_coords_vecs(coords, vecs, points_list):
    """
    Remove points that have been added from boundary from coordinates and their corresponding vectors.

    Parameters:
        coords (np.array): Array of coordinates.
        vecs (np.array): Array of vectors.
        points_list (list): List of coordinates (added from boundary for more accurate interpolation) to remove.

    Returns:
        np.array, np.array: Updated coords and vecs without the boundary points.
    """
    coords = np.array(coords)
    vecs = np.array(vecs)

    # Create a mask to keep only points **not** in points_list
    mask = np.array([not any(np.array_equal(coord, p) for p in points_list) for coord in coords])

    # Apply the mask to filter out unwanted coordinates and vectors
    coords_filtered = coords[mask]
    vecs_filtered = vecs[mask]

    return coords_filtered, vecs_filtered



# new fixed code

In [3]:
import numpy as np
from scipy.spatial import Delaunay

def interpolate(center, corners, vectors, eps_area=1e-12):
    """
    Robust barycentric interpolation: try Delaunay affine transform if possible,
    otherwise fall back to area-based absolute barycentric weights.
    corners: (3,2), center: (2,), vectors: (3,2)
    Returns interpolated vector (2,)
    """
    center = np.asarray(center, dtype=float)
    corners = np.asarray(corners, dtype=float)
    vectors = np.asarray(vectors, dtype=float)

    # Compute total area (using absolute cross)
    v0 = corners[1] - corners[0]
    v1 = corners[2] - corners[0]
    total_area = abs(np.cross(v0, v1)) / 2.0

    if total_area <= eps_area:
        # Degenerate triangle — avoid unstable division
        # Return average of available vectors as fallback (or zeros)
        return np.mean(vectors, axis=0)

    # Area-based barycentric weights (absolute areas of subtriangles)
    a0 = abs(np.cross(corners[1] - center, corners[2] - center)) / 2.0
    a1 = abs(np.cross(corners[0] - center, corners[2] - center)) / 2.0
    a2 = abs(np.cross(corners[0] - center, corners[1] - center)) / 2.0

    denom = a0 + a1 + a2
    if denom <= eps_area:
        # point is numerically problematic — fallback
        return np.mean(vectors, axis=0)

    w = np.array([a0, a1, a2]) / denom
    return w[0] * vectors[0] + w[1] * vectors[1] + w[2] * vectors[2]


def interpolate_zero_vectors(coords, vecs, points_list, debug=False, max_allowed_magnitude=1e4):
    """
    Interpolates vectors that are zero (except those in points_list). Uses Delaunay triangulation
    on valid points then computes barycentric interpolation per zero point. Adds safety checks.
    debug: when True prints info about problematic points.
    max_allowed_magnitude: any interpolated vector with magnitude above this will be replaced
        by a safe fallback (mean of neighbours).
    """
    coords = np.asarray(coords, dtype=float)
    vecs = np.asarray(vecs, dtype=float)

    # Convert point_list to a set of tuples for easy lookup
    point_set = {tuple(point) for point in points_list}

    # Identify indices of zero vectors (excluding those in points_list)
    zero_vec_indices = [i for i in range(len(vecs))
                        if np.allclose(vecs[i], 0.0) and tuple(coords[i]) not in point_set]

    # Indices of valid (non-zero) vectors
    valid_indices = [i for i in range(len(vecs)) if i not in zero_vec_indices]
    if len(valid_indices) < 3:
        # Not enough valid points to triangulate; fill zeros with mean of valid vectors
        if len(valid_indices) == 0:
            return vecs
        fallback = np.mean(vecs[valid_indices], axis=0)
        for i in zero_vec_indices:
            vecs[i] = fallback
        return vecs

    valid_coords = coords[valid_indices]
    valid_vecs = vecs[valid_indices]

    tri = Delaunay(valid_coords)

    # We'll use Delaunay.transform for barycentric coords when possible.
    for i in zero_vec_indices:
        p = coords[i]
        simplex_idx = tri.find_simplex(p)
        if simplex_idx == -1:
            # point outside triangulation: fallback to nearest-neighbour average
            # find k nearest valid points (k=3)
            dists = np.linalg.norm(valid_coords - p, axis=1)
            nn = np.argsort(dists)[:3]
            interp = np.mean(valid_vecs[nn], axis=0)
            vecs[i] = interp
            if debug:
                print(f"[outside] index {i}, point {p}, nearest dists {dists[nn]}")
            continue

        # tri.transform[simplex_idx] lets us compute barycentric coords without explicit division
        T = tri.transform[simplex_idx]
        # For 2D, transform has shape (3,2) or (2,2)+offset depending on SciPy version.
        # The usual method: r = dot(T[:2], p - T[2]); bary = [r0, r1, 1-r0-r1]
        try:
            r = np.dot(T[:2, :], p - T[2, :])
            bary = np.array([r[0], r[1], 1.0 - r[0] - r[1]])
            # map simplex vertex indices to original valid_indices
            vertex_indices = tri.simplices[simplex_idx]
            original_vertex_indices = [valid_indices[j] for j in vertex_indices]
            vectors_for_triangle = vecs[original_vertex_indices]  # these are non-zero by construction
            interp = bary[0] * vectors_for_triangle[0] + bary[1] * vectors_for_triangle[1] + bary[2] * vectors_for_triangle[2]
        except Exception:
            # fallback to area-based method using the triangle coordinates
            vertex_indices = tri.simplices[simplex_idx]
            original_vertex_indices = [valid_indices[j] for j in vertex_indices]
            corners = coords[original_vertex_indices]
            vectors_for_triangle = vecs[original_vertex_indices]
            interp = interpolate(p, corners, vectors_for_triangle)  # uses absolute-area fallback

        # sanity checks: large values indicate numerical instability
        mag = np.linalg.norm(interp)
        if not np.isfinite(mag) or mag > max_allowed_magnitude:
            # replace by safe fallback (mean of triangle vectors)
            if debug:
                print(f"[unstable] idx {i} point {p} mag={mag}. Using mean fallback.")
            interp = np.mean(vectors_for_triangle, axis=0)

        vecs[i] = interp

    return vecs


In [ ]:
'''
def interpolate_zero_vectors(coords, vecs, points_list):
    """Interpolates vectors for points with zero vectors, except those in points_list."""
    
    coords = np.array(coords)
    vecs = np.array(vecs)

    # Convert point_list to a set of tuples for easy lookup
    point_set = {tuple(point) for point in points_list}

    # Identify indices of zero vectors (excluding those in points_list)
    zero_vec_indices = [i for i in range(len(vecs)) if np.all(vecs[i] == 0) and tuple(coords[i]) not in point_set]

    # Get only valid points (excluding those with zero vectors)
    valid_indices = [i for i in range(len(vecs)) if i not in zero_vec_indices]
    valid_coords = coords[valid_indices]
    valid_vecs = vecs[valid_indices]

    # Create Delaunay triangulation only for valid coordinates
    tri = Delaunay(valid_coords)

    for i in zero_vec_indices:
        simplex_idx = tri.find_simplex(coords[i])

        if simplex_idx == -1:
            continue  # Skip if point is outside triangulation
        
        # Get the indices of the triangle's vertices (from valid points)
        vertex_indices = tri.simplices[simplex_idx]

        # Map back to original indices
        original_vertex_indices = [valid_indices[j] for j in vertex_indices]

        # Interpolate using non-zero vectors
        interpolated_vector = interpolate(coords[i], coords[original_vertex_indices], vecs[original_vertex_indices])
        vecs[i] = interpolated_vector

    return vecs



def interpolate(center, corners, vectors):
    """
    Interpolates a vector at `center` using surrounding `corners` and `vectors` via barycentric coordinates.

    Parameters:
        center (ndarray): The coordinate of the point to interpolate (1 x 2).
        corners (ndarray): The coordinates of surrounding points (3 x 2).
        vectors (ndarray): The vectors associated with the surrounding points (3 x 2).

    Returns:
        ndarray: Interpolated vector (1 x 2).
    """
    d0 = np.cross(corners[1] - center, corners[2] - center) / np.linalg.norm(corners[1] - corners[2])
    d1 = np.cross(corners[0] - center, corners[2] - center) / np.linalg.norm(corners[0] - corners[2])
    d2 = np.cross(corners[1] - center, corners[0] - center) / np.linalg.norm(corners[1] - corners[0])

    if (d0 + d1 + d2 == 0):
        return np.array([0, 0])

    s = np.array([d0, d1, d2]) / (d0 + d1 + d2)
    return s[0] * vectors[0] + s[1] * vectors[1] + s[2] * vectors[2]'''